# Retriever
- 비정형 질의(query)를 입력 받아 Vector store에서 관련된 내용을 검색하는 기능을 제공하는 인터페이스로 다양한 데이터 소스에서 정보를 검색하여 대규모 언어 모델(LLM) 기반 애플리케이션의 **정확성을** 향상시키는 데 핵심적인 역할을 한다.

![RAG](figures/rag2.png)

## 주요 특징
- **다양한 데이터 소스 지원**
	- Retriever는 벡터 스토어, 그래프 데이터베이스, 관계형 데이터베이스 등 여러 종류의 검색 시스템과 상호작용할 수 있는 통일된 인터페이스를 제공한다다.
- **간단한 인터페이스**: Retriever는 문자열 형태의 쿼리를 입력받아 관련 문서의 리스트를 반환한다. 이러한 단순한 구조 덕분에 다양한 검색 시스템과 쉽게 통합할 수 있다. 


## 다양한 Retriever 방식

**Retriever**란, 사용자의 질문(쿼리)에 가장 관련성 높은 정보를 찾아주는 구성 요소이다. 주로 검색 기반 질문응답 시스템(RAG, Retrieval-Augmented Generation)에서 사용된다. 다음은 자주 사용되는 다양한 Retriever의 유형과 그 특징이다.

1. **벡터 스토어(Vector Store) Retriever**
   - VectorStore로 부터 유사도를 기반으로 검색하는 가장 기본 Retriever
   - 텍스트 조각(청크)마다 **임베딩(embedding)을** 생성하여 벡터 공간에 저장하고, 쿼리 임베딩과의 **코사인 유사도(cosine similarity)** 등을 기반으로 유사한 텍스트를 검색한다.
   - 검색 속도가 빠르고 구현이 간단하여, 기본적인 검색 시스템을 구축할 때 적합하다.
2. **[ParentDocumentRetriever](https://python.langchain.com/docs/how_to/parent_document_retriever/)**
   - 하나의 문서를 여러 청크로 나눈 뒤 각각을 인덱싱하고, 쿼리와 가장 유사한 청크를 찾은 다음 해당 청크가 속한 **전체 원본 문서**를 반환한다.
   - 작은 정보 조각들이 모여 하나의 문서를 구성할 때 유용하며, 문맥을 넓게 유지할 수 있다.
3. **[MultiVectorRetriever](https://python.langchain.com/docs/how_to/multi_vector/)**
   - 각 문서에 대해 요약을 하거나, 가상의 질문을 생성하거나, 사람이 중요한 내용을 직접 추가하여 문서당 여러 개의 임베딩 벡터를 생성한다.
   - 텍스트 전체보다 더 핵심적인 정보가 검색에 반영되도록 하고자 할 때 효과적이다.
   - 특히, 문서가 길거나, 중요한 내용이 문서의 특정 부분에 집중되어 있는 경우에 유리하다.
4. **[SelfQueryRetriever](https://python.langchain.com/docs/how_to/self_query/)**
   - 대규모 언어 모델(LLM, Large Language Model)을 활용하여 사용자의 질문을 적절한 검색어와 **메타데이터(metadata)** 필터로 자동 변환한다.
   - 예를 들어, 문서의 작성자, 날짜, 태그와 같은 메타데이터를 기준으로 검색할 수 있다.
   - 문서 자체의 내용뿐만 아니라, 문서에 부가된 속성 정보에 대해 질문할 때 유용하다.
5. **[ContextualCompressionRetriever](https://python.langchain.com/docs/how_to/contextual_compression/)**
   - 기존 Retriever와 조합되어 사용된다.
   - 먼저 일반적인 검색을 수행한 후, 검색된 문서들에서 쿼리와 관련 없는 불필요한 정보를 제거하고 핵심 내용만을 추출하여 반환한다.
   - 정보를 요약하거나 압축하여 LLM에 전달할 문서 길이를 줄일 때 유용하다.
6. **[MultiQueryRetriever](https://python.langchain.com/docs/how_to/MultiQueryRetriever/)**
   - LLM을 이용해 하나의 쿼리로부터 여러 가지 변형된 쿼리를 생성하고, 각 쿼리에 대해 검색을 수행한 뒤 결과를 합치는 방식.
   - 다양한 표현에 강해 검색 범위를 넓히고 성능을 높인다.
7. **[EnsembleRetriever](https://python.langchain.com/docs/how_to/ensemble_retriever/)**
   - 여러 개의 Retriever(예: BM25, 벡터 기반 등)를 결합해 가중치를 기반으로 결과를 조합(re-ranking)한다.
   - 서로 다른 장점을 가진 Retriever를 하나로 묶어 성능을 강화한다.

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import MarkdownHeaderTextSplitter

from langchain_qdrant import QdrantVectorStore, FastEmbedSparse, RetrievalMode
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, SparseVectorParams, VectorParams, SparseIndexParams


loader = TextLoader("data/olympic_wiki.md", encoding="utf-8")

headers_to_split_on = [
    ("#", "H1"),
    ("##", "H2"),
    ("###", "H3"),
]

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on = headers_to_split_on
)

documents = loader.load()

docs = splitter.split_text('\n\n'.join([doc.page_content for doc in documents]))
print(len(docs))

In [ ]:
COLLECTION_NAME = "olympic_info"
VECTOR_SIZE = 3072

dense_embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
sparse_embeddings = FastEmbedSparse(model_name="Qdrant/bm25")

client = QdrantClient(host="localhost", port=6333)

if client.collection_exists(collection_name=COLLECTION_NAME):
    result = client.delete_collection(collection_name=COLLECTION_NAME)
    print("삭제:", result)


client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config={
        "dense": VectorParams(
            size=VECTOR_SIZE,
            distance=Distance.COSINE
        )
    },
    sparse_vectors_config={ 
        "sparse": SparseVectorParams(index=SparseIndexParams(on_disk=False))
    }
)

In [ ]:
vector_store = QdrantVectorStore(
    client=client,
    collection_name=COLLECTION_NAME,
   
    embedding=dense_embeddings,
    sparse_embedding=sparse_embeddings,
   
    retrieval_mode=RetrievalMode.HYBRID,
   
    vector_name="dense",
    sparse_vector_name="sparse",
)

vector_store.add_documents(documents=docs)

In [ ]:
retriever = vector_store.as_retriever()

result = retriever.invoke("1회 동계올림픽은 언제 어디에서 개최했지?")
print(len(result))
result

In [ ]:
retriever2 = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k":7,
        "score_threshold":0.7
    }
)
result = retriever2.invoke("올림픽과 관련된 논란들은 어떤 것들이 있었나?")
print(len(result))
result

In [ ]:
retriever3 = vector_store.as_retriever(
    search_type='mmr',
    search_kwargs={
        "k":5,
        "fetch_k": 10,
        "lambda_mult":0.5 
    }
)
result = retriever3.invoke("올림픽과 관련된 논란들은 어떤 것들이 있었나?")
print(len(result))
result

In [ ]:
##############################
# payload(metadata) filtering 
##############################
from qdrant_client.models import Filter, FieldCondition, MatchValue
filter_condition = Filter(
    must=[
        FieldCondition(key="metadata.H2", match=MatchValue(value="논란"))
    ]
)

retriever4 = vector_store.as_retriever(
    search_kwargs={
        "k":5,
        "filter":filter_condition
    }
)

result = retriever4.invoke("동계 올림픽 종목에는 무엇이있나요?") 

print(len(result))
result

In [ ]:
####################################################
#  invoke()할 때 Retriever의 설정들 동적으로 변경
####################################################

from langchain_core.runnables import ConfigurableField
retriever5 = retriever4.configurable_fields(

    search_type=ConfigurableField(
        id="search_type",
    ),
    search_kwargs=ConfigurableField(
        id="search_kwargs"
    )
)

In [ ]:
config = {
    "configurable":{
        "search_kwargs":{
            "k":10
        }
    }
}
query = "동계 올림픽 종목에는 무엇이있나요?"
retriever5.invoke(query, config=config)

In [ ]:
config = {
    "configurable": {
        "search_type":"mmr",
        "search_kwargs": {
            "k":5,
            "fetch_k":10,
            "lambda_mult": 0.2,
            # "filter":filter_condition
        }
    }
}
query = "올림픽에서 약물과 관련된 스캔들이 뭐가 있어지?"
retriever5.invoke(query, config=config)

In [ ]:
#############################################
# Retriever를 포함한 전체 chain 을 구성
#
# 쿼리 -> (retriever) -> 문서들/쿼리 -> (Prompt Template) -> prompt -> (llm) -> 응답
#############################################
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    template="""# Instruction:
당신은 정보제공을 목적으로하는 유능한 AI Assistant 입니다.
주어진 context의 내용을 기반으로 질문에 답변을 합니다.
Context에 질문에 대한 명확한 정보가 있는 경우 그것을 바탕으로 답변을 합니다.
Context에 질문에 대한 명확한 정보가 없는 경우 "정보가 부족해 답을 할 수없습니다." 라고 답합니다.
절대 추측이나 일반 상식을 바탕으로 답을 하거나 Context 없는 내용을 만들어서 답변해서는 안됩니다.

# Context:
{context}

# 질문:
{query}
"""
)

retriever = vector_store.as_retriever(search_kwargs={"k":5})
model = ChatOpenAI(model="gpt-5-mini")

def format_str(docs:list) -> str:
    """
    docs: list[Document] - vector store 검색한 내용에서 page_content만 추출해서 반환하는 Runnable
    """
    return '\n\n'.join(doc.page_content for doc in docs)

chain = {
    "context": retriever | format_str,
    "query":RunnablePassthrough()
} | prompt | model | StrOutputParser()

In [ ]:
query = "올림픽에서 약물과 관련된 스캔들이 뭐가 있어지?"
response = chain.invoke(query)

In [ ]:
print(response)